In [62]:
import numpy as np
import pandas as pd
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import load_img, img_to_array

import cv2


In [63]:
df = pd.read_csv('image_metadata.csv')
df

,file_path,experiment_id,position,trap_num,sample_id,treated,timepoint,species_label
0,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,0,P. aeruginosa
1,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,1,P. aeruginosa
2,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,2,P. aeruginosa
3,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,3,P. aeruginosa
4,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,4,P. aeruginosa
...,...,...,...,...,...,...,...,...
106838,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,281,20,3408-03,Treated,25,E. coli
106839,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,281,20,3408-03,Treated,26,E. coli
106840,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,281,20,3408-03,Treated,27,E. coli
106841,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6134 AST FISH 201126,281,20,3408-03,Treated,28,E. coli


In [64]:
filtered_df = df[(df['species_label'] != 'Unknown') & #only working with known species
                 (df['treated'] == 'Untreated') &  # removed treated antibiotic samples since the later timepoint images are harder to use
                 (df['timepoint'] == 30)] # looking at frame 15 for the 30 minute point
filtered_df

,file_path,experiment_id,position,trap_num,sample_id,treated,timepoint,species_label
30,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,16,43-16,Untreated,30,P. aeruginosa
64,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,30,43-08,Untreated,30,P. aeruginosa
98,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,40,43-18,Untreated,30,P. aeruginosa
132,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,101,41,43-19,Untreated,30,P. aeruginosa
166,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,102,7,123-07,Untreated,30,P. aeruginosa
...,...,...,...,...,...,...,...,...
82868,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6133 AST FISH 201125,178,2,5851-02,Untreated,30,E. faecalis
82903,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6133 AST FISH 201125,179,25,5919-08,Untreated,30,E. faecalis
82938,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6133 AST FISH 201125,180,5,5987-05,Untreated,30,E. faecalis
82973,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6133 AST FISH 201125,180,8,5987-08,Untreated,30,E. faecalis


In [65]:
filtered_df['species_label'].value_counts()

species_label
K. pneumoniae    788
E. coli          260
E. faecalis      230
P. aeruginosa    208
Name: count, dtype: int64

In [66]:
n = 200
filtered_df_sampled = filtered_df.groupby('species_label').apply(lambda x: x.sample(n=n, random_state=18)).reset_index(drop=True)
filtered_df_sampled #adjusted to 250 samples per species to account for majority class imbalance

C:\Users\silve\AppData\Local\Temp\ipykernel_22668\2582041369.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  filtered_df_sampled = filtered_df.groupby('species_label').apply(lambda x: x.sample(n=n, random_state=18)).reset_index(drop=True)


,file_path,experiment_id,position,trap_num,sample_id,treated,timepoint,species_label
0,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6131 AST FISH 201124,153,1,4403-01,Untreated,30,E. coli
1,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6131 AST FISH 201124,131,2,2643-02,Untreated,30,E. coli
2,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6131 AST FISH 201124,128,4,2403-04,Untreated,30,E. coli
3,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6129 AST FISH 201117,156,9,4185-09,Untreated,30,E. coli
4,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,121,25,1643-03,Untreated,30,E. coli
...,...,...,...,...,...,...,...,...
795,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6126 AST FISH 201116,189,23,7083-01,Untreated,30,P. aeruginosa
796,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6127 AST FISH 201117,162,4,4423-04,Untreated,30,P. aeruginosa
797,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6127 AST FISH 201117,178,23,5511-06,Untreated,30,P. aeruginosa
798,C:\Users\silve\Documents\210\data\Cropped-png-...,BV6129 AST FISH 201117,148,14,3641-14,Untreated,30,P. aeruginosa


In [67]:
def load_image(file_path, target_size=(64, 64)):
    img = load_img(file_path, target_size=target_size)
    img = img_to_array(img)
    img = img / 255.0  # Normalize the image
    return img

In [68]:
X = []
y = []

for _, row in filtered_df_sampled.iterrows():
    img_path = row['file_path']  # path to image file
    species_label = row['species_label']  # target variable
    
    image = load_image(img_path)
    X.append(image)
    
    species_map = {'E. faecalis': 0, 'K. pneumoniae': 1, 'E. coli': 2, 'P. aeruginosa': 3}
    y.append(species_map.get(species_label, -1))

In [69]:
X = np.array(X)
y = np.array(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=18, stratify=y)

In [70]:
print(f"Training set class distribution: {pd.Series(y_train).value_counts()}")
print(f"Test set class distribution: {pd.Series(y_test).value_counts()}")

Training set class distribution: 3    160
0    160
1    160
2    160
Name: count, dtype: int64
Test set class distribution: 0    40
2    40
1    40
3    40
Name: count, dtype: int64


In [71]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(4, activation='softmax')  # 4 species classes
])


model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

c:\Users\silve\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [72]:
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.2264 - loss: 1.5571 - val_accuracy: 0.2500 - val_loss: 1.3871
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.2562 - loss: 1.3874 - val_accuracy: 0.2500 - val_loss: 1.3865
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.2823 - loss: 1.3851 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.2405 - loss: 1.3866 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.2461 - loss: 1.3863 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.2146 - loss: 1.3864 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.2557 - loss: 1.3863 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.2286 - loss: 1.3864 - val_accuracy: 0.2500 - v

In [73]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print("Classification Report:\n", classification_report(y_test, y_pred_classes))
print("Accuracy Score: ", accuracy_score(y_test, y_pred_classes))

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
Classification Report:
               precision    recall  f1-score   support

           0       0.25      1.00      0.40        40
           1       0.00      0.00      0.00        40
           2       0.00      0.00      0.00        40
           3       0.00      0.00      0.00        40

    accuracy                           0.25       160
   macro avg       0.06      0.25      0.10       160
weighted avg       0.06      0.25      0.10       160

Accuracy Score:  0.25


c:\Users\silve\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\silve\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\silve\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [74]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

In [75]:
datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [76]:
model.fit(
    datagen.flow(X_train, y_train, batch_size=32),  # Use augmented data
    epochs=20,
    validation_data=(X_test, y_test),
)

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print("Classification Report:\n", classification_report(y_test, y_pred_classes))
print("Accuracy Score: ", accuracy_score(y_test, y_pred_classes))

Epoch 1/20


c:\Users\silve\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 65ms/step - accuracy: 0.2381 - loss: 1.3863 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 83ms/step - accuracy: 0.2623 - loss: 1.3863 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 112ms/step - accuracy: 0.2373 - loss: 1.3864 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 126ms/step - accuracy: 0.2771 - loss: 1.3863 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 95ms/step - accuracy: 0.2399 - loss: 1.3864 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 147ms/step - accuracy: 0.2562 - loss: 1.3863 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 7/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step - accuracy: 0.2435 - loss: 1.3864 - val_accuracy: 0.2500 - val_loss: 1.3863
Epoch 8/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 69ms/step - accuracy: 0.2155 - loss: 1.3864 - val_accuracy: 0.2500 - val_loss:

c:\Users\silve\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\silve\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\silve\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

# more preprocessing

In [ ]:
import imageio
import numpy as np
import sys
sys.path.insert(0, os.path.expanduser("scripts"))
import imgtool as t
import albumentations as A

In [ ]:
def preprocess_image(image_path, target_size=(1300, 52)):
    img = imageio.imread(image_path).astype("float32")
    img = t.pad_or_crop(t.nm(img), target_size)
    return img

In [ ]:
def select_frames(files, num_frames = 15, is_testset = False, test_frame_idx = 0):
    files = files[:num_frames]
    while len(files) < num_frames:  # Duplicate last frame if needed
        files.append(files[-1])
    
    if is_testset:
        return files[test_frame_idx:test_frame_idx + num_frames]
    else:
        start_idx = np.random.randint(len(files) - num_frames + 1)
        return files[start_idx:start_idx + num_frames]

In [ ]:
def augment_images(images, train_aug, test_aug, is_testset):
    transform = A.Compose(test_aug if is_testset else train_aug)
    transformed_images = [transform(image=img)['image'] for img in images]
    return np.stack(transformed_images)


# simplified preporcessing and nn 

In [1]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import imageio
import albumentations as A
import cv2
from glob import glob

c:\Users\silve\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [2]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Define directories
DATA_FOLDER = "/data"
LABEL_FOLDER = "/data/selections"

In [4]:
def load_and_preprocess_image(image_path, expect_shape=(1300, 52)):
    image = imageio.imread(image_path).astype("float32")
    image = cv2.resize(image, (expect_shape[1], expect_shape[0]))  # Resize
    return image / 255.0  # Normalize

def phase_from_location(trap_location, expect_shape=(1300, 52), expect_num_frames=32):
    files = sorted(glob(os.path.join(trap_location, "phase*.png")))[:expect_num_frames]

    # Duplicate last frame if needed
    while len(files) < expect_num_frames:
        files.append(files[-1])

    img_list = [load_and_preprocess_image(f, expect_shape) for f in files]
    img_arr = np.stack(img_list)  # Shape: (frames, height, width)
    return img_arr

# Get dataset manually
def get_dataset(data_folder, label_folder, train_frac=0.8):
    labels = ["488", "CY3", "cy5", "txr"]
    all_traps = []
    
    for fluor_lbl in labels:
        label_files = glob(os.path.join(label_folder, f"selected_traps*EXP*/single_species_loc_{fluor_lbl}"))
        for label_file in label_files:
            with open(label_file) as f:
                trap_locations = f.read().splitlines()
            all_traps.extend([(os.path.join(data_folder, trap), fluor_lbl) for trap in trap_locations])
    
    # Train-test split
    random.shuffle(all_traps)
    split_idx = int(train_frac * len(all_traps))
    train_data, test_data = all_traps[:split_idx], all_traps[split_idx:]
    return train_data, test_data

In [3]:
class BacteriaCNN(nn.Module):
    def __init__(self):
        super(BacteriaCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(128 * 1300 * 52, 256)
        self.fc2 = nn.Linear(256, 4)  # 4 classes

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        x = torch.flatten(x, start_dim=1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [5]:
train_data, test_data = get_dataset(DATA_FOLDER, LABEL_FOLDER)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BacteriaCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10
batch_size = 8

In [9]:
for epoch in range(num_epochs):
    model.train()
    random.shuffle(train_data)

    total_loss = 0
    for i in range(0, len(train_data), batch_size):
        batch = train_data[i:i + batch_size]
        images, labels = [], []

        for trap_location, label in batch:
            img_arr = phase_from_location(trap_location)  # Shape: (frames, height, width)
            img_arr = img_arr[:1]  # Use only the first frame
            img_tensor = torch.tensor(img_arr, dtype=torch.float32).unsqueeze(0).to(device)  # Add channel dim
            images.append(img_tensor)

            labels.append(["488", "CY3", "cy5", "txr"].index(label))

        images = torch.cat(images, dim=0)  # (batch, 1, height, width)
        labels = torch.tensor(labels, dtype=torch.long).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss} / {len(train_data)}")

print("Training complete!")

Epoch 1/10, Loss: 0 / 0
Epoch 2/10, Loss: 0 / 0
Epoch 3/10, Loss: 0 / 0
Epoch 4/10, Loss: 0 / 0
Epoch 5/10, Loss: 0 / 0
Epoch 6/10, Loss: 0 / 0
Epoch 7/10, Loss: 0 / 0
Epoch 8/10, Loss: 0 / 0
Epoch 9/10, Loss: 0 / 0
Epoch 10/10, Loss: 0 / 0
Training complete!
